# Paper Figures

Generates the figures shown in the paper from saved `.npz` result files.
Run `02_train.ipynb` first (or use the pre-trained results in `results/`).

**Run from the repository root directory.**

BEST CASE

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import resample
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
import os.path
# Import your custom functions (make sure these .py files are in the same directory)
from models import PhysicsInformedLSTM2, PhysicsInformedLSTM_Contributions
from utils import normalize, calculate_nrmse
from SDOF_aceleracion_promedio_tf import SDOF_aceleracion_promedio_tf
# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# ==============================================================================
# 1. DEFINE ALL INFERENCE CONFIGURATIONS
# ==============================================================================
# Each dictionary in this list represents one complete inference run.
# Add or modify these configurations as needed.
CONFIGURATIONS = [
    {
        "seismic_input": "BI-BNCS_CNP100",
        "feature_vector_filename": "lstm_features_CNP100V2.npy",
        "N_MODES": 3,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL",
        "start_sec": 2.0,
        "end_sec": 39.0,
    },
    {
        "seismic_input": "BI-BNCS_SP100",
        "feature_vector_filename": "lstm_features_SP100V2.npy", # Example filename
        "N_MODES": 3, # Example: this run might have used 3 modes
        "N_DOF": 6,
        "current_hidden_dim": 128, # Example: different hidden dim
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL", # Example: different init
        "start_sec": 6.0,
        "end_sec": 171.0,
    },
    {
        "seismic_input": "BI-BNCS_ICA140",
        "feature_vector_filename": "lstm_features_ICA140V2.npy",
        "N_MODES": 3,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL",
        "start_sec": 10.0,
        "end_sec": 179.0,
    }
]
# Constants that are the same for all runs
WINDOW_SIZE = 100
Fs = 100
ORIGINAL_FS = 200
# Constants
N_DOF = 6
M = tf.convert_to_tensor(np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5 + 406.6)]).astype(np.float32))
R = tf.convert_to_tensor(np.ones((N_DOF, 1), dtype=np.float32))
M_np = M.numpy()
R_np = R.numpy()
# ==============================================================================
# 2. INFERENCE AND HELPER FUNCTIONS
# ==============================================================================
@tf.function
def run_inference_window(
    w_tf,
    modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
    modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
    modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
    hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
    current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm,
    p_n1, model_mode1, model_mode2, model_mode3, model_contrib,
    max_acc, max_displ, Fs_tf, R, N_MODES, N_DOF): # Pass N_MODES and N_DOF
    """
    Performs a forward pass for one window, mirroring the training logic.
    """
    # STAGE 1: PRELIMINARY SDOF SOLVE
    interp_freq_mode1_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode1, lambda: tf.repeat(2.0, WINDOW_SIZE))
    interp_damp_mode1_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode1, lambda: tf.repeat(0.1, WINDOW_SIZE))
    kt_mode1_prev = (2.0 * np.pi * interp_freq_mode1_prev)**2
    x_mode1_prev, _, a_int_mode1_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode1_prev, interp_damp_mode1_prev, p_n1, Fs_tf, x0=modal_y_prev_mode1, v0=modal_ydot_prev_mode1)
    a_int_mode2_prev, x_mode2_prev = (tf.zeros_like(a_int_mode1_prev), tf.zeros_like(x_mode1_prev))
    if N_MODES >= 2:
        interp_freq_mode2_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode2, lambda: tf.repeat(2.0, WINDOW_SIZE))
        interp_damp_mode2_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode2, lambda: tf.repeat(0.1, WINDOW_SIZE))
        kt_mode2_prev = (2.0 * np.pi * interp_freq_mode2_prev)**2
        x_mode2_prev, _, a_int_mode2_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode2_prev, interp_damp_mode2_prev, p_n1, Fs_tf, x0=modal_y_prev_mode2, v0=modal_ydot_prev_mode2)
   
    a_int_mode3_prev, x_mode3_prev = (tf.zeros_like(a_int_mode1_prev), tf.zeros_like(x_mode1_prev))
    if N_MODES >= 3:
        interp_freq_mode3_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode3, lambda: tf.repeat(5.0, WINDOW_SIZE))
        interp_damp_mode3_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode3, lambda: tf.repeat(0.02, WINDOW_SIZE))
        kt_mode3_prev = (2.0 * np.pi * interp_freq_mode3_prev)**2
        x_mode3_prev, _, a_int_mode3_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode3_prev, interp_damp_mode3_prev, p_n1, Fs_tf, x0=modal_y_prev_mode3, v0=modal_ydot_prev_mode3)
    # STAGE 2: CONSTRUCT FEATURES & GET PREDICTIONS
    window_feat_mode1 = tf.stack([current_seismic_norm, a_int_mode1_prev / max_acc, x_mode1_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode1), normalize(prev_freq_seq_mode1)], axis=1)[tf.newaxis, :, :]
    pred_out_mode1, hidden_state_mode1 = model_mode1(window_feat_mode1, initial_state=hidden_state_mode1, training=False)
    new_damp_seq_mode1, new_freq_seq_mode1 = pred_out_mode1[0, :, 0], pred_out_mode1[0, :, 1]
   
    new_damp_seq_mode2, new_freq_seq_mode2 = prev_damp_seq_mode2, prev_freq_seq_mode2
    if N_MODES >= 2 and model_mode2 is not None:
        window_feat_mode2 = tf.stack([current_seismic_norm, a_int_mode2_prev / max_acc, x_mode2_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode2), normalize(prev_freq_seq_mode2)], axis=1)[tf.newaxis, :, :]
        pred_out_mode2, hidden_state_mode2 = model_mode2(window_feat_mode2, initial_state=hidden_state_mode2, training=False)
        new_damp_seq_mode2, new_freq_seq_mode2 = pred_out_mode2[0, :, 0], pred_out_mode2[0, :, 1]
    new_damp_seq_mode3, new_freq_seq_mode3 = prev_damp_seq_mode3, prev_freq_seq_mode3
    if N_MODES >= 3 and model_mode3 is not None:
        window_feat_mode3 = tf.stack([current_seismic_norm, a_int_mode3_prev / max_acc, x_mode3_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode3), normalize(prev_freq_seq_mode3)], axis=1)[tf.newaxis, :, :]
        pred_out_mode3, hidden_state_mode3 = model_mode3(window_feat_mode3, initial_state=hidden_state_mode3, training=False)
        new_damp_seq_mode3, new_freq_seq_mode3 = pred_out_mode3[0, :, 0], pred_out_mode3[0, :, 1]
   
    phi_mode1_t_trans, phi_mode2_t_trans, phi_mode3_t_trans = (tf.zeros([N_DOF, WINDOW_SIZE]), tf.zeros([N_DOF, WINDOW_SIZE]), tf.zeros([N_DOF, WINDOW_SIZE]))
    if N_DOF > 1 and model_contrib is not None:
        features = [current_seismic_norm, a_int_mode1_prev/max_acc, x_mode1_prev/max_displ, normalize(prev_damp_seq_mode1), normalize(prev_freq_seq_mode1)]
        if N_MODES >= 2: features.extend([a_int_mode2_prev/max_acc, x_mode2_prev/max_displ, normalize(prev_damp_seq_mode2), normalize(prev_freq_seq_mode2)])
        if N_MODES >= 3: features.extend([a_int_mode3_prev/max_acc, x_mode3_prev/max_displ, normalize(prev_damp_seq_mode3), normalize(prev_freq_seq_mode3)])
        features.extend([current_cnn_norm, next_seismic_norm, next_cnn_norm])
        window_feat_contrib = tf.stack(features, axis=1)[tf.newaxis, :, :]
        pred_phi, hidden_state_contrib = model_contrib(window_feat_contrib, initial_state=hidden_state_contrib, training=False)
        phi_mode1_t_trans = tf.transpose(pred_phi[0, :, 0*N_DOF:1*N_DOF])
        if N_MODES >= 2: phi_mode2_t_trans = tf.transpose(pred_phi[0, :, 1*N_DOF:2*N_DOF])
        if N_MODES >= 3: phi_mode3_t_trans = tf.transpose(pred_phi[0, :, 2*N_DOF:3*N_DOF])
    elif N_DOF == 1:
        phi_mode1_t_trans = tf.ones([1, WINDOW_SIZE])
    # STAGE 3: FINAL SDOF SOLVE & RESPONSE CALCULATION
    kt_mode1 = (2.0 * np.pi * new_freq_seq_mode1)**2
    x_mode1, v_mode1, a_int_mode1 = SDOF_aceleracion_promedio_tf(1.0, kt_mode1, new_damp_seq_mode1, p_n1, Fs_tf, x0=modal_y_prev_mode1, v0=modal_ydot_prev_mode1)
   
    a_int_mode2, x_mode2, v_mode2 = (tf.zeros_like(a_int_mode1), tf.zeros_like(x_mode1), tf.zeros_like(v_mode1))
    if N_MODES >= 2:
        kt_mode2 = (2.0 * np.pi * new_freq_seq_mode2)**2
        x_mode2, v_mode2, a_int_mode2 = SDOF_aceleracion_promedio_tf(1.0, kt_mode2, new_damp_seq_mode2, p_n1, Fs_tf, x0=modal_y_prev_mode2, v0=modal_ydot_prev_mode2)
    a_int_mode3, x_mode3, v_mode3 = (tf.zeros_like(a_int_mode1), tf.zeros_like(x_mode1), tf.zeros_like(v_mode1))
    if N_MODES >= 3:
        kt_mode3 = (2.0 * np.pi * new_freq_seq_mode3)**2
        x_mode3, v_mode3, a_int_mode3 = SDOF_aceleracion_promedio_tf(1.0, kt_mode3, new_damp_seq_mode3, p_n1, Fs_tf, x0=modal_y_prev_mode3, v0=modal_ydot_prev_mode3)
    mode1_contrib_acc = phi_mode1_t_trans * tf.expand_dims(a_int_mode1, axis=0)
    mode2_contrib_acc = phi_mode2_t_trans * tf.expand_dims(a_int_mode2, axis=0)
    mode3_contrib_acc = phi_mode3_t_trans * tf.expand_dims(a_int_mode3, axis=0)
    a_total_n1 = mode1_contrib_acc + mode2_contrib_acc + mode3_contrib_acc + tf.expand_dims(-p_n1, axis=0) * R
    mode1_contrib_displ = phi_mode1_t_trans * tf.expand_dims(x_mode1, axis=0)
    mode2_contrib_displ = phi_mode2_t_trans * tf.expand_dims(x_mode2, axis=0)
    mode3_contrib_displ = phi_mode3_t_trans * tf.expand_dims(x_mode3, axis=0)
    displ_total_n1 = mode1_contrib_displ + mode2_contrib_displ + mode3_contrib_displ
   
    # STAGE 4: UPDATE STATES
    new_modal_y_prev_mode1, new_modal_ydot_prev_mode1 = x_mode1[-1], v_mode1[-1]
    new_modal_y_prev_mode2, new_modal_ydot_prev_mode2 = x_mode2[-1], v_mode2[-1]
    new_modal_y_prev_mode3, new_modal_ydot_prev_mode3 = x_mode3[-1], v_mode3[-1]
    return (
        new_modal_y_prev_mode1, new_modal_ydot_prev_mode1, new_damp_seq_mode1, new_freq_seq_mode1,
        new_modal_y_prev_mode2, new_modal_ydot_prev_mode2, new_damp_seq_mode2, new_freq_seq_mode2,
        new_modal_y_prev_mode3, new_modal_ydot_prev_mode3, new_damp_seq_mode3, new_freq_seq_mode3,
        hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
        a_total_n1, displ_total_n1, phi_mode1_t_trans, phi_mode2_t_trans, phi_mode3_t_trans,
        a_int_mode1, a_int_mode2, a_int_mode3, x_mode1, x_mode2, x_mode3, v_mode1, v_mode2, v_mode3
    )
def run_single_inference(config):
    """
    Runs the entire inference process for a single configuration and returns results.
    """
    print("-" * 50)
    print(f"Running inference for: {config['seismic_input']}")
   
    # Unpack config
    seismic_input = config['seismic_input']
    feature_vector_filename = config['feature_vector_filename']
    N_MODES = config['N_MODES']
    N_DOF = config['N_DOF']
    current_hidden_dim = config['current_hidden_dim']
    start_sec, end_sec = config['start_sec'], config['end_sec']
    # Generate folder and weight paths
    weight_str = f"acc{round(100*config['ACC_LOSS_WEIGHT'])}_disp{round(100*config['DISPL_LOSS_WEIGHT'])}"
    folder_name = f"{seismic_input}_{weight_str}_NM{N_MODES}_NDOF{N_DOF}_{config['init_str']}_Hiddim{current_hidden_dim}"
    save_dir = os.path.join(os.getcwd(), folder_name)
    weights_path_mode1 = os.path.join(save_dir, f"best_model_mode1_{folder_name}.weights.h5")
    weights_path_mode2 = os.path.join(save_dir, f"best_model_mode2_{folder_name}.weights.h5")
    weights_path_mode3 = os.path.join(save_dir, f"best_model_mode3_{folder_name}.weights.h5")
    weights_path_contrib = os.path.join(save_dir, f"best_model_contrib_{folder_name}.weights.h5")
    # Data Loading
    data_path = os.path.join(os.getcwd(), seismic_input)
    try:
        accel_data = [np.loadtxt(os.path.join(data_path, f"a{i}.txt")) for i in range(1, N_DOF + 1)]
        displ_data = [np.loadtxt(os.path.join(data_path, f"d{i}.txt")) for i in range(1, N_DOF + 1)]
        u = np.loadtxt(os.path.join(data_path, "u.txt"))
        cnn_features = np.load(os.path.join(data_path, feature_vector_filename))
        du = np.loadtxt(os.path.join(data_path, "du.txt"))
    except FileNotFoundError as e:
        print(f"Error loading data for {seismic_input}: {e}")
        return None
    # Data Processing
    accel_all = np.stack(accel_data[::-1], axis=1)
    displ_all = np.stack(displ_data[::-1], axis=1)
    new_timesteps = len(u) // (ORIGINAL_FS // Fs)
    accel_all, u, displ_all, du = resample(accel_all, new_timesteps, axis=0), resample(u, new_timesteps), resample(displ_all, new_timesteps, axis=0), resample(du, new_timesteps)
    start_idx, end_idx = int(start_sec * Fs), min(int(end_sec * Fs), len(u))
    accel_all, u, displ_all, du = accel_all[start_idx:end_idx], u[start_idx:end_idx], displ_all[start_idx:end_idx], du[start_idx:end_idx]
    cnn_features = cnn_features[start_idx // WINDOW_SIZE : end_idx // WINDOW_SIZE]
    # Normalization
    max_displ, max_acc, max_seismic = np.max(np.abs(displ_all)), np.max(np.abs(accel_all)), np.max(np.abs(u))
    max_cnn = np.max(np.abs(cnn_features)) if cnn_features.size > 0 else 1.0
    ground_acc_t_norm = tf.convert_to_tensor(u / max_seismic, dtype=tf.float32)
    cnn_features_norm = cnn_features / max_cnn if cnn_features.size > 0 else np.zeros_like(cnn_features)
    # Model Initialization and Weight Loading
    model_mode1 = PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=1)
    model_mode2 = None if N_MODES < 2 else PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=2)
    model_mode3 = None if N_MODES < 3 else PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=3)
    contrib_input_dim = 4 + 4*N_MODES
    model_contrib = None if N_DOF == 1 else PhysicsInformedLSTM_Contributions(input_dim=contrib_input_dim, hidden_dim=current_hidden_dim, n_dof=N_DOF * N_MODES)
   
    # Build models
    dummy_input = tf.zeros((1, WINDOW_SIZE, 8))
    _ = model_mode1(dummy_input)
    if model_mode2: _ = model_mode2(dummy_input)
    if model_mode3: _ = model_mode3(dummy_input)
    if model_contrib: _ = model_contrib(tf.zeros((1, WINDOW_SIZE, contrib_input_dim)))
    # Load weights
    try:
        model_mode1.load_weights(weights_path_mode1)
        if N_MODES >= 2: model_mode2.load_weights(weights_path_mode2)
        if N_MODES >= 3: model_mode3.load_weights(weights_path_mode3)
        if N_DOF > 1: model_contrib.load_weights(weights_path_contrib)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights for {folder_name}: {e}")
        return None
    # Inference Loop
    total_steps = len(u)
    num_windows = total_steps // WINDOW_SIZE
    predictable_windows = num_windows - 1 if num_windows > 1 else 0
    if predictable_windows <= 0: return None
   
    # Precompute constants and matrices
    PHI_full = np.array([
        [1.0000, 1.0000, 1.0000], [0.9868, 0.7725, 0.1319], [0.9706, 0.4449, -0.76728],
        [0.9451, -0.0509, -0.9784], [0.9202, -0.5931, -0.2761], [0.8804, -0.9093, 0.4327]
    ]).astype(np.float32)[:N_DOF, :3]
    M_np = np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5+406.6)][:N_DOF]).astype(np.float32)
    R = tf.constant(np.ones((N_DOF, 1), dtype=np.float32))
    PHI = tf.constant(PHI_full)
    # Initialize states
    modal_y_prev_mode1, modal_ydot_prev_mode1 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode1, prev_freq_seq_mode1 = tf.repeat(0.1, WINDOW_SIZE), tf.repeat(2.0, WINDOW_SIZE)
    modal_y_prev_mode2, modal_ydot_prev_mode2 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode2, prev_freq_seq_mode2 = tf.repeat(0.1, WINDOW_SIZE), tf.repeat(2.0, WINDOW_SIZE)
    modal_y_prev_mode3, modal_ydot_prev_mode3 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode3, prev_freq_seq_mode3 = tf.repeat(0.02, WINDOW_SIZE), tf.repeat(5.0, WINDOW_SIZE)
    hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib = None, None, None, None
   
    all_pred_acc, all_pred_displ, all_damp, all_freq, all_pred_phi = [], [], [[] for _ in range(3)], [[] for _ in range(3)], [[] for _ in range(3)]
    for w in range(predictable_windows):
        start, end = w * WINDOW_SIZE, (w + 1) * WINDOW_SIZE
        next_start, next_end = (w + 1) * WINDOW_SIZE, (w + 2) * WINDOW_SIZE
        current_seismic_norm, current_cnn_norm = ground_acc_t_norm[start:end], tf.convert_to_tensor(cnn_features_norm[w], dtype=tf.float32)
        next_seismic_norm, next_cnn_norm = ground_acc_t_norm[next_start:next_end], tf.convert_to_tensor(cnn_features_norm[w+1], dtype=tf.float32)
        p_n1 = -tf.convert_to_tensor(u[next_start:next_end], dtype=tf.float32)
        (modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
         modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
         modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
         hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
         a_total_n1, displ_total_n1, phi1_t, phi2_t, phi3_t,
         _, _, _, _, _, _, _, _, _) = run_inference_window(
            tf.constant(w), modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
            modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
            modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
            hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
            current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm,
            p_n1, model_mode1, model_mode2, model_mode3, model_contrib,
            tf.constant(max_acc, dtype=tf.float32),
            tf.constant(max_displ, dtype=tf.float32),
            tf.constant(float(Fs)), R, N_MODES, N_DOF
        )
        all_pred_acc.append(a_total_n1.numpy())
        all_pred_displ.append(displ_total_n1.numpy())
        if N_MODES >= 1:
            all_damp[0].append(prev_damp_seq_mode1.numpy()); all_freq[0].append(prev_freq_seq_mode1.numpy()); all_pred_phi[0].append(phi1_t.numpy())
        if N_MODES >= 2:
            all_damp[1].append(prev_damp_seq_mode2.numpy()); all_freq[1].append(prev_freq_seq_mode2.numpy()); all_pred_phi[1].append(phi2_t.numpy())
        if N_MODES >= 3:
            all_damp[2].append(prev_damp_seq_mode3.numpy()); all_freq[2].append(prev_freq_seq_mode3.numpy()); all_pred_phi[2].append(phi3_t.numpy())
    # Post-processing
    pred_acc_np = np.concatenate(all_pred_acc, axis=1)
    pred_displ_np = np.concatenate(all_pred_displ, axis=1)
    damp_np = [np.concatenate(d) for d in all_damp if d]
    freq_np = [np.concatenate(f) for f in all_freq if f]
    pred_phi_np = [np.concatenate(p, axis=1) for p in all_pred_phi if p]
    valid_steps = pred_acc_np.shape[1]
    true_acc_np = accel_all.T[:, WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    true_displ_np = displ_all.T[:, WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    time_axis = np.arange(valid_steps) / Fs  # Start at 0, adjusted to remove WINDOW_SIZE offset
    ground_motion_u = u[WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    # Save results to .npz file
    output_path = os.path.join(save_dir, f"results_{folder_name}.npz")
    np.savez(output_path,
            pred_acc=pred_acc_np,
            true_acc=true_acc_np,
            pred_displ=pred_displ_np,
            true_displ=true_displ_np,
            damp=damp_np,
            freq=freq_np,
            phi=pred_phi_np,
            time=time_axis,
            ground_motion=ground_motion_u,
            config=config)
    print(f"\nResults saved to: {output_path}")

    return {
        "config": config, "pred_acc": pred_acc_np, "true_acc": true_acc_np,
        "pred_displ": pred_displ_np, "true_displ": true_displ_np,
        "damp": damp_np, "freq": freq_np, "phi": pred_phi_np,
        "time": time_axis, "ground_motion": ground_motion_u
    }
# ==============================================================================
# 3. REFINED PLOTTING FUNCTIONS
# ==============================================================================
def get_rounded_limits_and_ticks(data_min, data_max, num_ticks=5):
    if data_min == data_max: data_max += 1e-6
    data_range = data_max - data_min
    power = 10**np.floor(np.log10(data_range))
    possible_steps = np.array([0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 2.0, 2.5, 5.0])
    best_step = possible_steps[np.argmin(np.abs(data_range / (power * possible_steps) - num_ticks))] * power
    new_min = np.floor(data_min / best_step) * best_step
    new_max = np.ceil(data_max / best_step) * best_step
    ticks = np.arange(new_min, new_max + best_step/2, best_step)
    return (new_min, new_max), ticks
def plot_concatenated_modal_params(concatenated_data, max_modes, separator_times, all_results, plot_type='Frequency'):
    # --- Adjust figure height and row heights ---
    fig, axs = plt.subplots(min(2, max_modes) + 1, 1, figsize=(8.5, 5.5), gridspec_kw={'height_ratios': [1, 1, 0.5]}, sharex=True)  # Half height for ground motion
   
    time = concatenated_data["time"]
    if plot_type == 'Frequency': data_key, color = "freq", '#000080'
    else: data_key, color = "damp", "#0678AD"
    plot_idx = 0
    for i in range(min(2, max_modes) - 1, -1, -1):  # Limit to modes 1 and 2
        ax = axs[plot_idx]
        data = concatenated_data[data_key][i]
       
        if plot_type == 'Damping':
            data = data * 100
            ax.set_ylabel(fr"$\xi_{{{i+1}}}$ (%)")
            ax.set_ylim(0, 50)
            ax.set_yticks(np.arange(0, 51, 10))
        else:  # Frequency
            ax.set_ylabel(fr"$f_{{{i+1}}}$ (Hz)")
            if i == 0:
                ax.set_ylim(0, 1.5)
                ax.set_yticks([0, 0.5, 1, 1.5])  # Specific ticks
            elif i == 1:
                ax.set_ylim(1, 5)  # Changed limits to 1 to 5
                ax.set_yticks([1, 2, 3, 4, 5])  # Specific ticks
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'))
        ax.plot(time, data, color=color, linewidth=1.2)
        ax.grid(True, linestyle=':')
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
        plot_idx += 1
    ax_gm = axs[-1]
    time_offset = 0; colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for i, result in enumerate(all_results):
        run_time = result['time'] - result['time'][0] + time_offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=colors[i])  # Convert to g
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max(concatenated_data['ground_motion'] / 9.81) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', ''), ha='center', fontsize=10)
        time_offset = run_time[-1] + (1/Fs)
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
   
    ax_gm.set_xlim(0, time[-1])
    ymin, ymax = ax_gm.get_ylim(); y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max); ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
def plot_individual_mode_shapes(results):
    channel_map = {1: "1", 3: "3", 6: "6"}  # Changed "Roof" to "1" and "3rd Floor" to "3"
    channels_to_plot = [0, 2, 5]
    mode_colors = ['#6a0dad', '#20b2aa']  # Limit to modes 1 and 2
    
    # Create 4x1 subplot with adjusted height ratios (3 full rows + half-height ground motion)
    fig, axs = plt.subplots(4, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1, 1, 0.5]}, sharex=True, squeeze=True)
    
    # Legend for modes 1 and 2
    handles = [Line2D([0], [0], color=c, label=f'Mode {i+1}') for i, c in enumerate(mode_colors)]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
    
    # Calculate total concatenated time and separator times
    total_time = 0
    separator_times = []
    for i, result in enumerate(results):
        duration = len(result['time']) / Fs  # Duration in seconds
        total_time += duration
        if i < len(results) - 1:
            separator_times.append(total_time - (1 / (2 * Fs)))  # Midpoint between end of current and start of next
    
    # Plot mode shapes for each channel with separators
    for row, ch_idx in enumerate(channels_to_plot):
        ax = axs[row]
        channel_name_latex = channel_map[ch_idx + 1].replace(' ', r'\ ')
        time_offset = 0
        for i, result in enumerate(results):
            run_time = result['time'] - result['time'][0] + time_offset  # Relative time + cumulative offset
            for mode_idx in range(2):  # Limit to modes 1 and 2
                ax.plot(run_time, result['phi'][mode_idx][ch_idx, :], color=mode_colors[mode_idx], linewidth=1.2)
            time_offset += len(result['time']) / Fs  # Add duration of current result
        ax.set_ylabel(fr"$\phi_{{{channel_name_latex}}}$")
        ax.grid(True, linestyle=':')
        ax.set_xlim(0, total_time)
        ax.set_ylim(-1.5, 2.5)  # Uniform y-limits for all mode shape plots
        ax.set_yticks([-1.5, -0.5, 0.5, 1.5, 2.5])  # Specific ticks
        ax.set_xticks(np.linspace(0, total_time, 5))
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)  # Add separators
    
    # Plot ground motion
    ax_gm = axs[3]
    time_offset = 0
    for i, result in enumerate(results):
        run_time = result['time'] - result['time'][0] + time_offset  # Relative time + cumulative offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=f"C{i}")  # Convert to g
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max([r['ground_motion'].max() / 9.81 for r in results]) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', '')[0:6], ha='center', fontsize=8)  # Shortened labels
        time_offset += len(result['time']) / Fs  # Add duration of current result
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    ax_gm.set_xlim(0, total_time)
    ymin, ymax = ax_gm.get_ylim()
    y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max)
    ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))
    ax_gm.set_xticks(np.linspace(0, total_time, 5))
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
    
    plt.tight_layout(rect=[0.15, 0, 0.85, 0.96])  # Adjusted to accommodate decorations
    plt.show()
def round_to_nearest(value, base=0.05):
    """Round a value to the nearest multiple of base."""
    return np.round(value / base) * base
def plot_hysteresis_comparison(results):
    global M_np, R_np
    num_events = len(results)
    # Set figure size to make plots square: 3.5 inches per subplot
    fig, axs = plt.subplots(1, num_events, figsize=(3.5 * num_events, 3.5), squeeze=False, sharey=False)
   
    handles = [Line2D([0], [0], color='#000080', lw=2.0, label='True'),
               Line2D([0], [0], color="#FF0015", lw=1.0, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=True, edgecolor='black')
   
    for col, result in enumerate(results):
        ax = axs[0, col]
        # Compute base shear in kN (divide by 1000), convert acceleration to g
        Qb_true = np.dot(R_np.T, np.dot(M_np, result['true_acc'] / 9.81)) / 1000.0
        Qb_pred = np.dot(R_np.T, np.dot(M_np, result['pred_acc'] / 9.81)) / 1000.0
        # Displacement in meters with negative sign
        d_iso_true = -result['true_displ'][result['config']['N_DOF']-1, :]
        d_iso_pred = -result['pred_displ'][result['config']['N_DOF']-1, :]
       
        ax.plot(d_iso_true, Qb_true[0, :], color='#000080', linewidth=1.5)
        ax.plot(d_iso_pred, Qb_pred[0, :], '--', color="#FF0015", linewidth=1.0)
       
        ax.set_title(result['config']['seismic_input'].replace('BI-BNCS_', ''))
        ax.set_xlabel("Isolation Drift (m)")
        if col == 0:
            ax.set_ylabel("Base Shear (kN)")
       
        # Set x-axis: centered at 0, symmetrical, 5 ticks rounded to 0.05
        x_min = min(np.min(d_iso_true), np.min(d_iso_pred))
        x_max = max(np.max(d_iso_true), np.max(d_iso_pred))
        x_abs_max = round_to_nearest(max(abs(x_min), abs(x_max)), base=0.05)
        ax.set_xlim(-x_abs_max, x_abs_max)
        x_ticks = np.linspace(-x_abs_max, x_abs_max, 5)
        x_ticks = [round_to_nearest(x, base=0.05) for x in x_ticks]
        ax.set_xticks(x_ticks)
       
        # Set y-axis: centered at 0, symmetrical, 5 ticks with middle at 0, rounded to 0.05
        y_min = min(np.min(Qb_true), np.min(Qb_pred))
        y_max = max(np.max(Qb_true), np.max(Qb_pred))
        y_abs_max = round_to_nearest(max(abs(y_min), abs(y_max)), base=0.05)
        ax.set_ylim(-y_abs_max, y_abs_max)
        y_ticks = np.linspace(-y_abs_max, y_abs_max, 5)
        y_ticks = [round_to_nearest(y, base=0.05) for y in y_ticks]
        ax.set_yticks(y_ticks)
       
        # Set major and minor grid: 4x4 minor grid lines
        ax.grid(True, which='major', linestyle=':', linewidth=0.5)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.5)
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(x_abs_max / 4))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(y_abs_max / 4))
   
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()
def plot_isolation_layer_acceleration(results):
    """Plot isolation layer accelerations for CNP100, SP100, ICA140 in a single column."""
    channel_map = {6: "6"}
    ch_idx = 5 # DOF 6 (isolation layer, zero-based index)
    num_rows = 3 # One row per seismic input (CNP100, SP100, ICA140)
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)
   
    # Shared legend
    handles = [Line2D([0], [0], color="C0", lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
   
    # Map seismic inputs to row indices
    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}
   
    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue
       
        time = result['time']
        true_data = result['true_acc'][ch_idx, :] / 9.81  # Convert to g
        pred_data = result['pred_acc'][ch_idx, :] / 9.81  # Convert to g
        nrmse = calculate_nrmse(pred_data, true_data)
       
        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)
       
        # Add seismic input name inside a text box
        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_ylabel(r"$\ddot{u}_{6}$ (g)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))  # Ensure 5 uniform ticks
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])  # 2 decimal places
        ax.set_xticks(np.linspace(time[0], time[-1], 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()
def plot_isolation_layer_displacement(results):
    """Plot isolation layer displacements for CNP100, SP100, ICA140 in a single column."""
    channel_map = {6: "6"}
    ch_idx = 5 # DOF 6 (isolation layer, zero-based index)
    num_rows = 3 # One row per seismic input (CNP100, SP100, ICA140)
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)
   
    # Shared legend
    handles = [Line2D([0], [0], color='C0', lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
   
    # Map seismic inputs to row indices
    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}
   
    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue
       
        time = result['time']
        true_data = result['true_displ'][ch_idx, :]
        pred_data = result['pred_displ'][ch_idx, :]
        nrmse = calculate_nrmse(pred_data, true_data)
       
        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)
       
        # Add seismic input name inside a text box
        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_ylabel(r"$u_{6}$ (m)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))  # Ensure 5 uniform ticks
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])  # 2 decimal places
        ax.set_xticks(np.linspace(time[0], time[-1], 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()
   
# ==============================================================================
# 5. MAIN EXECUTION
# ==============================================================================
def main():
    # Set global font style for all plots
    plt.rcParams['font.style'] = 'italic'
    plt.rcParams['font.size'] = 10
   
    all_results = []
    for config in CONFIGURATIONS:
        result = run_single_inference(config)
        if result:
            all_results.append(result)
    if not all_results:
        print("No results to plot.")
        return
   
    # --- Plotting ---
    plot_isolation_layer_acceleration(all_results)
    plot_isolation_layer_displacement(all_results)
   
    # Concatenate data for modal plots
    concatenated = {}
    separator_times = []
    concatenated["ground_motion"] = np.concatenate([r["ground_motion"] for r in all_results])
    max_modes = max(r['config']['N_MODES'] for r in all_results)
    for key in ["damp", "freq"]:
        padded_runs = []
        for r in all_results:
            run_data = r[key]
            padding = [np.full_like(run_data[0], np.nan) for _ in range(max_modes - len(run_data))]
            padded_runs.append(run_data + padding)
        concatenated[key] = [np.concatenate([run[i] for run in padded_runs]) for i in range(max_modes)]
    time_arrays = []
    current_time = 0
    for r in all_results:
        run_duration = r['time'][-1] - r['time'][0]
        time_arrays.append(np.linspace(current_time, current_time + run_duration, len(r['time'])))
        current_time += run_duration + (1/Fs)
        if len(separator_times) < len(all_results) - 1:
            separator_times.append(current_time - (1/Fs)/2)
    concatenated["time"] = np.concatenate(time_arrays)
   
    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Frequency')
    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Damping')
    plot_individual_mode_shapes(all_results)
    plot_hysteresis_comparison(all_results)
if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import resample
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
import os.path
# Import your custom functions (make sure these .py files are in the same directory)
from models import PhysicsInformedLSTM2, PhysicsInformedLSTM_Contributions
from utils import normalize, calculate_nrmse
from SDOF_aceleracion_promedio_tf import SDOF_aceleracion_promedio_tf
# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# ==============================================================================
# 1. DEFINE ALL INFERENCE CONFIGURATIONS
# ==============================================================================
# Each dictionary in this list represents one complete inference run.
# Add or modify these configurations as needed.
CONFIGURATIONS = [
    {
        "seismic_input": "BI-BNCS_CNP100",
        "feature_vector_filename": "lstm_features_CNP100V2.npy",
        "N_MODES": 2,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 0,
        "init_str": "FINAL",
        "start_sec": 2.0,
        "end_sec": 39.0,
    },
    {
        "seismic_input": "BI-BNCS_SP100",
        "feature_vector_filename": "lstm_features_SP100V2.npy", # Example filename
        "N_MODES": 2, # Example: this run might have used 3 modes
        "N_DOF": 6,
        "current_hidden_dim": 128, # Example: different hidden dim
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 0,
        "init_str": "FINAL", # Example: different init
        "start_sec": 6.0,
        "end_sec": 171.0,
    },
    {
        "seismic_input": "BI-BNCS_ICA140",
        "feature_vector_filename": "lstm_features_ICA140V2.npy",
        "N_MODES": 2,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 0,
        "init_str": "FINAL",
        "start_sec": 10.0,
        "end_sec": 179.0,
    }
]
# Constants that are the same for all runs
WINDOW_SIZE = 100
Fs = 100
ORIGINAL_FS = 200
# Constants
N_DOF = 6
M = tf.convert_to_tensor(np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5 + 406.6)]).astype(np.float32))
R = tf.convert_to_tensor(np.ones((N_DOF, 1), dtype=np.float32))
M_np = M.numpy()
R_np = R.numpy()
# ==============================================================================
# 2. INFERENCE AND HELPER FUNCTIONS
# ==============================================================================
@tf.function
def run_inference_window(
    w_tf,
    modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
    modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
    modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
    hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
    current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm,
    p_n1, model_mode1, model_mode2, model_mode3, model_contrib,
    max_acc, max_displ, Fs_tf, R, N_MODES, N_DOF): # Pass N_MODES and N_DOF
    """
    Performs a forward pass for one window, mirroring the training logic.
    """
    # STAGE 1: PRELIMINARY SDOF SOLVE
    interp_freq_mode1_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode1, lambda: tf.repeat(2.0, WINDOW_SIZE))
    interp_damp_mode1_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode1, lambda: tf.repeat(0.1, WINDOW_SIZE))
    kt_mode1_prev = (2.0 * np.pi * interp_freq_mode1_prev)**2
    x_mode1_prev, _, a_int_mode1_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode1_prev, interp_damp_mode1_prev, p_n1, Fs_tf, x0=modal_y_prev_mode1, v0=modal_ydot_prev_mode1)
    a_int_mode2_prev, x_mode2_prev = (tf.zeros_like(a_int_mode1_prev), tf.zeros_like(x_mode1_prev))
    if N_MODES >= 2:
        interp_freq_mode2_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode2, lambda: tf.repeat(2.0, WINDOW_SIZE))
        interp_damp_mode2_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode2, lambda: tf.repeat(0.1, WINDOW_SIZE))
        kt_mode2_prev = (2.0 * np.pi * interp_freq_mode2_prev)**2
        x_mode2_prev, _, a_int_mode2_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode2_prev, interp_damp_mode2_prev, p_n1, Fs_tf, x0=modal_y_prev_mode2, v0=modal_ydot_prev_mode2)
   
    a_int_mode3_prev, x_mode3_prev = (tf.zeros_like(a_int_mode1_prev), tf.zeros_like(x_mode1_prev))
    if N_MODES >= 3:
        interp_freq_mode3_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_freq_seq_mode3, lambda: tf.repeat(5.0, WINDOW_SIZE))
        interp_damp_mode3_prev = tf.cond(tf.greater(w_tf, 0), lambda: prev_damp_seq_mode3, lambda: tf.repeat(0.02, WINDOW_SIZE))
        kt_mode3_prev = (2.0 * np.pi * interp_freq_mode3_prev)**2
        x_mode3_prev, _, a_int_mode3_prev = SDOF_aceleracion_promedio_tf(1.0, kt_mode3_prev, interp_damp_mode3_prev, p_n1, Fs_tf, x0=modal_y_prev_mode3, v0=modal_ydot_prev_mode3)
    # STAGE 2: CONSTRUCT FEATURES & GET PREDICTIONS
    window_feat_mode1 = tf.stack([current_seismic_norm, a_int_mode1_prev / max_acc, x_mode1_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode1), normalize(prev_freq_seq_mode1)], axis=1)[tf.newaxis, :, :]
    pred_out_mode1, hidden_state_mode1 = model_mode1(window_feat_mode1, initial_state=hidden_state_mode1, training=False)
    new_damp_seq_mode1, new_freq_seq_mode1 = pred_out_mode1[0, :, 0], pred_out_mode1[0, :, 1]
   
    new_damp_seq_mode2, new_freq_seq_mode2 = prev_damp_seq_mode2, prev_freq_seq_mode2
    if N_MODES >= 2 and model_mode2 is not None:
        window_feat_mode2 = tf.stack([current_seismic_norm, a_int_mode2_prev / max_acc, x_mode2_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode2), normalize(prev_freq_seq_mode2)], axis=1)[tf.newaxis, :, :]
        pred_out_mode2, hidden_state_mode2 = model_mode2(window_feat_mode2, initial_state=hidden_state_mode2, training=False)
        new_damp_seq_mode2, new_freq_seq_mode2 = pred_out_mode2[0, :, 0], pred_out_mode2[0, :, 1]
    new_damp_seq_mode3, new_freq_seq_mode3 = prev_damp_seq_mode3, prev_freq_seq_mode3
    if N_MODES >= 3 and model_mode3 is not None:
        window_feat_mode3 = tf.stack([current_seismic_norm, a_int_mode3_prev / max_acc, x_mode3_prev / max_displ, current_cnn_norm, next_seismic_norm, next_cnn_norm, normalize(prev_damp_seq_mode3), normalize(prev_freq_seq_mode3)], axis=1)[tf.newaxis, :, :]
        pred_out_mode3, hidden_state_mode3 = model_mode3(window_feat_mode3, initial_state=hidden_state_mode3, training=False)
        new_damp_seq_mode3, new_freq_seq_mode3 = pred_out_mode3[0, :, 0], pred_out_mode3[0, :, 1]
   
    phi_mode1_t_trans, phi_mode2_t_trans, phi_mode3_t_trans = (tf.zeros([N_DOF, WINDOW_SIZE]), tf.zeros([N_DOF, WINDOW_SIZE]), tf.zeros([N_DOF, WINDOW_SIZE]))
    if N_DOF > 1 and model_contrib is not None:
        features = [current_seismic_norm, a_int_mode1_prev/max_acc, x_mode1_prev/max_displ, normalize(prev_damp_seq_mode1), normalize(prev_freq_seq_mode1)]
        if N_MODES >= 2: features.extend([a_int_mode2_prev/max_acc, x_mode2_prev/max_displ, normalize(prev_damp_seq_mode2), normalize(prev_freq_seq_mode2)])
        if N_MODES >= 3: features.extend([a_int_mode3_prev/max_acc, x_mode3_prev/max_displ, normalize(prev_damp_seq_mode3), normalize(prev_freq_seq_mode3)])
        features.extend([current_cnn_norm, next_seismic_norm, next_cnn_norm])
        window_feat_contrib = tf.stack(features, axis=1)[tf.newaxis, :, :]
        pred_phi, hidden_state_contrib = model_contrib(window_feat_contrib, initial_state=hidden_state_contrib, training=False)
        phi_mode1_t_trans = tf.transpose(pred_phi[0, :, 0*N_DOF:1*N_DOF])
        if N_MODES >= 2: phi_mode2_t_trans = tf.transpose(pred_phi[0, :, 1*N_DOF:2*N_DOF])
        if N_MODES >= 3: phi_mode3_t_trans = tf.transpose(pred_phi[0, :, 2*N_DOF:3*N_DOF])
    elif N_DOF == 1:
        phi_mode1_t_trans = tf.ones([1, WINDOW_SIZE])
    # STAGE 3: FINAL SDOF SOLVE & RESPONSE CALCULATION
    kt_mode1 = (2.0 * np.pi * new_freq_seq_mode1)**2
    x_mode1, v_mode1, a_int_mode1 = SDOF_aceleracion_promedio_tf(1.0, kt_mode1, new_damp_seq_mode1, p_n1, Fs_tf, x0=modal_y_prev_mode1, v0=modal_ydot_prev_mode1)
   
    a_int_mode2, x_mode2, v_mode2 = (tf.zeros_like(a_int_mode1), tf.zeros_like(x_mode1), tf.zeros_like(v_mode1))
    if N_MODES >= 2:
        kt_mode2 = (2.0 * np.pi * new_freq_seq_mode2)**2
        x_mode2, v_mode2, a_int_mode2 = SDOF_aceleracion_promedio_tf(1.0, kt_mode2, new_damp_seq_mode2, p_n1, Fs_tf, x0=modal_y_prev_mode2, v0=modal_ydot_prev_mode2)
    a_int_mode3, x_mode3, v_mode3 = (tf.zeros_like(a_int_mode1), tf.zeros_like(x_mode1), tf.zeros_like(v_mode1))
    if N_MODES >= 3:
        kt_mode3 = (2.0 * np.pi * new_freq_seq_mode3)**2
        x_mode3, v_mode3, a_int_mode3 = SDOF_aceleracion_promedio_tf(1.0, kt_mode3, new_damp_seq_mode3, p_n1, Fs_tf, x0=modal_y_prev_mode3, v0=modal_ydot_prev_mode3)
    mode1_contrib_acc = phi_mode1_t_trans * tf.expand_dims(a_int_mode1, axis=0)
    mode2_contrib_acc = phi_mode2_t_trans * tf.expand_dims(a_int_mode2, axis=0)
    mode3_contrib_acc = phi_mode3_t_trans * tf.expand_dims(a_int_mode3, axis=0)
    a_total_n1 = mode1_contrib_acc + mode2_contrib_acc + mode3_contrib_acc + tf.expand_dims(-p_n1, axis=0) * R
    mode1_contrib_displ = phi_mode1_t_trans * tf.expand_dims(x_mode1, axis=0)
    mode2_contrib_displ = phi_mode2_t_trans * tf.expand_dims(x_mode2, axis=0)
    mode3_contrib_displ = phi_mode3_t_trans * tf.expand_dims(x_mode3, axis=0)
    displ_total_n1 = mode1_contrib_displ + mode2_contrib_displ + mode3_contrib_displ
   
    # STAGE 4: UPDATE STATES
    new_modal_y_prev_mode1, new_modal_ydot_prev_mode1 = x_mode1[-1], v_mode1[-1]
    new_modal_y_prev_mode2, new_modal_ydot_prev_mode2 = x_mode2[-1], v_mode2[-1]
    new_modal_y_prev_mode3, new_modal_ydot_prev_mode3 = x_mode3[-1], v_mode3[-1]
    return (
        new_modal_y_prev_mode1, new_modal_ydot_prev_mode1, new_damp_seq_mode1, new_freq_seq_mode1,
        new_modal_y_prev_mode2, new_modal_ydot_prev_mode2, new_damp_seq_mode2, new_freq_seq_mode2,
        new_modal_y_prev_mode3, new_modal_ydot_prev_mode3, new_damp_seq_mode3, new_freq_seq_mode3,
        hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
        a_total_n1, displ_total_n1, phi_mode1_t_trans, phi_mode2_t_trans, phi_mode3_t_trans,
        a_int_mode1, a_int_mode2, a_int_mode3, x_mode1, x_mode2, x_mode3, v_mode1, v_mode2, v_mode3
    )
def run_single_inference(config):
    """
    Runs the entire inference process for a single configuration and returns results.
    """
    print("-" * 50)
    print(f"Running inference for: {config['seismic_input']}")
   
    # Unpack config
    seismic_input = config['seismic_input']
    feature_vector_filename = config['feature_vector_filename']
    N_MODES = config['N_MODES']
    N_DOF = config['N_DOF']
    current_hidden_dim = config['current_hidden_dim']
    start_sec, end_sec = config['start_sec'], config['end_sec']
    # Generate folder and weight paths
    weight_str = f"acc{round(100*config['ACC_LOSS_WEIGHT'])}_disp{round(100*config['DISPL_LOSS_WEIGHT'])}"
    folder_name = f"{seismic_input}_{weight_str}_NM{N_MODES}_NDOF{N_DOF}_{config['init_str']}_Hiddim{current_hidden_dim}"
    save_dir = os.path.join(os.getcwd(), folder_name)
    weights_path_mode1 = os.path.join(save_dir, f"best_model_mode1_{folder_name}.weights.h5")
    weights_path_mode2 = os.path.join(save_dir, f"best_model_mode2_{folder_name}.weights.h5")
    weights_path_mode3 = os.path.join(save_dir, f"best_model_mode3_{folder_name}.weights.h5")
    weights_path_contrib = os.path.join(save_dir, f"best_model_contrib_{folder_name}.weights.h5")
    # Data Loading
    data_path = os.path.join(os.getcwd(), seismic_input)
    try:
        accel_data = [np.loadtxt(os.path.join(data_path, f"a{i}.txt")) for i in range(1, N_DOF + 1)]
        displ_data = [np.loadtxt(os.path.join(data_path, f"d{i}.txt")) for i in range(1, N_DOF + 1)]
        u = np.loadtxt(os.path.join(data_path, "u.txt"))
        cnn_features = np.load(os.path.join(data_path, feature_vector_filename))
        du = np.loadtxt(os.path.join(data_path, "du.txt"))
    except FileNotFoundError as e:
        print(f"Error loading data for {seismic_input}: {e}")
        return None
    # Data Processing
    accel_all = np.stack(accel_data[::-1], axis=1)
    displ_all = np.stack(displ_data[::-1], axis=1)
    new_timesteps = len(u) // (ORIGINAL_FS // Fs)
    accel_all, u, displ_all, du = resample(accel_all, new_timesteps, axis=0), resample(u, new_timesteps), resample(displ_all, new_timesteps, axis=0), resample(du, new_timesteps)
    start_idx, end_idx = int(start_sec * Fs), min(int(end_sec * Fs), len(u))
    accel_all, u, displ_all, du = accel_all[start_idx:end_idx], u[start_idx:end_idx], displ_all[start_idx:end_idx], du[start_idx:end_idx]
    cnn_features = cnn_features[start_idx // WINDOW_SIZE : end_idx // WINDOW_SIZE]
    # Normalization
    max_displ, max_acc, max_seismic = np.max(np.abs(displ_all)), np.max(np.abs(accel_all)), np.max(np.abs(u))
    max_cnn = np.max(np.abs(cnn_features)) if cnn_features.size > 0 else 1.0
    ground_acc_t_norm = tf.convert_to_tensor(u / max_seismic, dtype=tf.float32)
    cnn_features_norm = cnn_features / max_cnn if cnn_features.size > 0 else np.zeros_like(cnn_features)
    # Model Initialization and Weight Loading
    model_mode1 = PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=1)
    model_mode2 = None if N_MODES < 2 else PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=2)
    model_mode3 = None if N_MODES < 3 else PhysicsInformedLSTM2(input_dim=8, hidden_dim=current_hidden_dim, mode=3)
    contrib_input_dim = 4 + 4*N_MODES
    model_contrib = None if N_DOF == 1 else PhysicsInformedLSTM_Contributions(input_dim=contrib_input_dim, hidden_dim=current_hidden_dim, n_dof=N_DOF * N_MODES)
   
    # Build models
    dummy_input = tf.zeros((1, WINDOW_SIZE, 8))
    _ = model_mode1(dummy_input)
    if model_mode2: _ = model_mode2(dummy_input)
    if model_mode3: _ = model_mode3(dummy_input)
    if model_contrib: _ = model_contrib(tf.zeros((1, WINDOW_SIZE, contrib_input_dim)))
    # Load weights
    try:
        model_mode1.load_weights(weights_path_mode1)
        if N_MODES >= 2: model_mode2.load_weights(weights_path_mode2)
        if N_MODES >= 3: model_mode3.load_weights(weights_path_mode3)
        if N_DOF > 1: model_contrib.load_weights(weights_path_contrib)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights for {folder_name}: {e}")
        return None
    # Inference Loop
    total_steps = len(u)
    num_windows = total_steps // WINDOW_SIZE
    predictable_windows = num_windows - 1 if num_windows > 1 else 0
    if predictable_windows <= 0: return None
   
    # Precompute constants and matrices
    PHI_full = np.array([
        [1.0000, 1.0000, 1.0000], [0.9868, 0.7725, 0.1319], [0.9706, 0.4449, -0.76728],
        [0.9451, -0.0509, -0.9784], [0.9202, -0.5931, -0.2761], [0.8804, -0.9093, 0.4327]
    ]).astype(np.float32)[:N_DOF, :3]
    M_np = np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5+406.6)][:N_DOF]).astype(np.float32)
    R = tf.constant(np.ones((N_DOF, 1), dtype=np.float32))
    PHI = tf.constant(PHI_full)
    # Initialize states
    modal_y_prev_mode1, modal_ydot_prev_mode1 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode1, prev_freq_seq_mode1 = tf.repeat(0.1, WINDOW_SIZE), tf.repeat(2.0, WINDOW_SIZE)
    modal_y_prev_mode2, modal_ydot_prev_mode2 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode2, prev_freq_seq_mode2 = tf.repeat(0.1, WINDOW_SIZE), tf.repeat(2.0, WINDOW_SIZE)
    modal_y_prev_mode3, modal_ydot_prev_mode3 = tf.constant(0.0), tf.constant(0.0)
    prev_damp_seq_mode3, prev_freq_seq_mode3 = tf.repeat(0.02, WINDOW_SIZE), tf.repeat(5.0, WINDOW_SIZE)
    hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib = None, None, None, None
   
    all_pred_acc, all_pred_displ, all_damp, all_freq, all_pred_phi = [], [], [[] for _ in range(3)], [[] for _ in range(3)], [[] for _ in range(3)]
    for w in range(predictable_windows):
        start, end = w * WINDOW_SIZE, (w + 1) * WINDOW_SIZE
        next_start, next_end = (w + 1) * WINDOW_SIZE, (w + 2) * WINDOW_SIZE
        current_seismic_norm, current_cnn_norm = ground_acc_t_norm[start:end], tf.convert_to_tensor(cnn_features_norm[w], dtype=tf.float32)
        next_seismic_norm, next_cnn_norm = ground_acc_t_norm[next_start:next_end], tf.convert_to_tensor(cnn_features_norm[w+1], dtype=tf.float32)
        p_n1 = -tf.convert_to_tensor(u[next_start:next_end], dtype=tf.float32)
        (modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
         modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
         modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
         hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
         a_total_n1, displ_total_n1, phi1_t, phi2_t, phi3_t,
         _, _, _, _, _, _, _, _, _) = run_inference_window(
            tf.constant(w), modal_y_prev_mode1, modal_ydot_prev_mode1, prev_damp_seq_mode1, prev_freq_seq_mode1,
            modal_y_prev_mode2, modal_ydot_prev_mode2, prev_damp_seq_mode2, prev_freq_seq_mode2,
            modal_y_prev_mode3, modal_ydot_prev_mode3, prev_damp_seq_mode3, prev_freq_seq_mode3,
            hidden_state_mode1, hidden_state_mode2, hidden_state_mode3, hidden_state_contrib,
            current_seismic_norm, current_cnn_norm, next_seismic_norm, next_cnn_norm,
            p_n1, model_mode1, model_mode2, model_mode3, model_contrib,
            tf.constant(max_acc, dtype=tf.float32),
            tf.constant(max_displ, dtype=tf.float32),
            tf.constant(float(Fs)), R, N_MODES, N_DOF
        )
        all_pred_acc.append(a_total_n1.numpy())
        all_pred_displ.append(displ_total_n1.numpy())
        if N_MODES >= 1:
            all_damp[0].append(prev_damp_seq_mode1.numpy()); all_freq[0].append(prev_freq_seq_mode1.numpy()); all_pred_phi[0].append(phi1_t.numpy())
        if N_MODES >= 2:
            all_damp[1].append(prev_damp_seq_mode2.numpy()); all_freq[1].append(prev_freq_seq_mode2.numpy()); all_pred_phi[1].append(phi2_t.numpy())
        if N_MODES >= 3:
            all_damp[2].append(prev_damp_seq_mode3.numpy()); all_freq[2].append(prev_freq_seq_mode3.numpy()); all_pred_phi[2].append(phi3_t.numpy())
    # Post-processing
    pred_acc_np = np.concatenate(all_pred_acc, axis=1)
    pred_displ_np = np.concatenate(all_pred_displ, axis=1)
    damp_np = [np.concatenate(d) for d in all_damp if d]
    freq_np = [np.concatenate(f) for f in all_freq if f]
    pred_phi_np = [np.concatenate(p, axis=1) for p in all_pred_phi if p]
    valid_steps = pred_acc_np.shape[1]
    true_acc_np = accel_all.T[:, WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    true_displ_np = displ_all.T[:, WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    time_axis = np.arange(valid_steps) / Fs  # Start at 0, adjusted to remove WINDOW_SIZE offset
    ground_motion_u = u[WINDOW_SIZE : WINDOW_SIZE + valid_steps]
    # Save results to .npz file
    output_path = os.path.join(save_dir, f"results_{folder_name}.npz")
    np.savez(output_path,
            pred_acc=pred_acc_np,
            true_acc=true_acc_np,
            pred_displ=pred_displ_np,
            true_displ=true_displ_np,
            damp=damp_np,
            freq=freq_np,
            phi=pred_phi_np,
            time=time_axis,
            ground_motion=ground_motion_u,
            config=config)
    print(f"\nResults saved to: {output_path}")

    return {
        "config": config, "pred_acc": pred_acc_np, "true_acc": true_acc_np,
        "pred_displ": pred_displ_np, "true_displ": true_displ_np,
        "damp": damp_np, "freq": freq_np, "phi": pred_phi_np,
        "time": time_axis, "ground_motion": ground_motion_u
    }
# ==============================================================================
# 3. REFINED PLOTTING FUNCTIONS
# ==============================================================================
def get_rounded_limits_and_ticks(data_min, data_max, num_ticks=5):
    if data_min == data_max: data_max += 1e-6
    data_range = data_max - data_min
    power = 10**np.floor(np.log10(data_range))
    possible_steps = np.array([0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 2.0, 2.5, 5.0])
    best_step = possible_steps[np.argmin(np.abs(data_range / (power * possible_steps) - num_ticks))] * power
    new_min = np.floor(data_min / best_step) * best_step
    new_max = np.ceil(data_max / best_step) * best_step
    ticks = np.arange(new_min, new_max + best_step/2, best_step)
    return (new_min, new_max), ticks
def plot_concatenated_modal_params(concatenated_data, max_modes, separator_times, all_results, plot_type='Frequency'):
    # --- Adjust figure height and row heights ---
    fig, axs = plt.subplots(min(2, max_modes) + 1, 1, figsize=(8.5, 5.5), gridspec_kw={'height_ratios': [1, 1, 0.5]}, sharex=True)  # Half height for ground motion
   
    time = concatenated_data["time"]
    if plot_type == 'Frequency': data_key, color = "freq", '#000080'
    else: data_key, color = "damp", "#0678AD"
    plot_idx = 0
    for i in range(min(2, max_modes) - 1, -1, -1):  # Limit to modes 1 and 2
        ax = axs[plot_idx]
        data = concatenated_data[data_key][i]
       
        if plot_type == 'Damping':
            data = data * 100
            ax.set_ylabel(fr"$\xi_{{{i+1}}}$ (%)")
            ax.set_ylim(0, 50)
            ax.set_yticks(np.arange(0, 51, 10))
        else:  # Frequency
            ax.set_ylabel(fr"$f_{{{i+1}}}$ (Hz)")
            if i == 0:
                ax.set_ylim(0, 1.5)
                ax.set_yticks([0, 0.5, 1, 1.5])  # Specific ticks
            elif i == 1:
                ax.set_ylim(1, 5)  # Changed limits to 1 to 5
                ax.set_yticks([1, 2, 3, 4, 5])  # Specific ticks
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'))
        ax.plot(time, data, color=color, linewidth=1.2)
        ax.grid(True, linestyle=':')
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
        plot_idx += 1
    ax_gm = axs[-1]
    time_offset = 0; colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for i, result in enumerate(all_results):
        run_time = result['time'] - result['time'][0] + time_offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=colors[i])  # Convert to g
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max(concatenated_data['ground_motion'] / 9.81) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', ''), ha='center', fontsize=10)
        time_offset = run_time[-1] + (1/Fs)
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
   
    ax_gm.set_xlim(0, time[-1])
    ymin, ymax = ax_gm.get_ylim(); y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max); ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
def plot_individual_mode_shapes(results):
    channel_map = {1: "1", 3: "3", 6: "6"}  # Changed "Roof" to "1" and "3rd Floor" to "3"
    channels_to_plot = [0, 2, 5]
    mode_colors = ['#6a0dad', '#20b2aa']  # Limit to modes 1 and 2
    
    # Create 4x1 subplot with adjusted height ratios (3 full rows + half-height ground motion)
    fig, axs = plt.subplots(4, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1, 1, 0.5]}, sharex=True, squeeze=True)
    
    # Legend for modes 1 and 2
    handles = [Line2D([0], [0], color=c, label=f'Mode {i+1}') for i, c in enumerate(mode_colors)]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
    
    # Calculate total concatenated time and separator times
    total_time = 0
    separator_times = []
    for i, result in enumerate(results):
        duration = len(result['time']) / Fs  # Duration in seconds
        total_time += duration
        if i < len(results) - 1:
            separator_times.append(total_time - (1 / (2 * Fs)))  # Midpoint between end of current and start of next
    
    # Plot mode shapes for each channel with separators
    for row, ch_idx in enumerate(channels_to_plot):
        ax = axs[row]
        channel_name_latex = channel_map[ch_idx + 1].replace(' ', r'\ ')
        time_offset = 0
        for i, result in enumerate(results):
            run_time = result['time'] - result['time'][0] + time_offset  # Relative time + cumulative offset
            for mode_idx in range(2):  # Limit to modes 1 and 2
                ax.plot(run_time, result['phi'][mode_idx][ch_idx, :], color=mode_colors[mode_idx], linewidth=1.2)
            time_offset += len(result['time']) / Fs  # Add duration of current result
        ax.set_ylabel(fr"$\phi_{{{channel_name_latex}}}$")
        ax.grid(True, linestyle=':')
        ax.set_xlim(0, total_time)
        ax.set_ylim(-1.5, 2.5)  # Uniform y-limits for all mode shape plots
        ax.set_yticks([-1.5, -0.5, 0.5, 1.5, 2.5])  # Specific ticks
        ax.set_xticks(np.linspace(0, total_time, 5))
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)  # Add separators
    
    # Plot ground motion
    ax_gm = axs[3]
    time_offset = 0
    for i, result in enumerate(results):
        run_time = result['time'] - result['time'][0] + time_offset  # Relative time + cumulative offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=f"C{i}")  # Convert to g
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max([r['ground_motion'].max() / 9.81 for r in results]) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', '')[0:6], ha='center', fontsize=8)  # Shortened labels
        time_offset += len(result['time']) / Fs  # Add duration of current result
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    ax_gm.set_xlim(0, total_time)
    ymin, ymax = ax_gm.get_ylim()
    y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max)
    ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))
    ax_gm.set_xticks(np.linspace(0, total_time, 5))
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
    
    plt.tight_layout(rect=[0.15, 0, 0.85, 0.96])  # Adjusted to accommodate decorations
    plt.show()
def round_to_nearest(value, base=0.05):
    """Round a value to the nearest multiple of base."""
    return np.round(value / base) * base
def plot_hysteresis_comparison(results):
    global M_np, R_np
    num_events = len(results)
    # Set figure size to make plots square: 3.5 inches per subplot
    fig, axs = plt.subplots(1, num_events, figsize=(3.5 * num_events, 3.5), squeeze=False, sharey=False)
   
    handles = [Line2D([0], [0], color='#000080', lw=2.0, label='True'),
               Line2D([0], [0], color="#FF0015", lw=1.0, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=True, edgecolor='black')
   
    for col, result in enumerate(results):
        ax = axs[0, col]
        # Compute base shear in kN (divide by 1000), convert acceleration to g
        Qb_true = np.dot(R_np.T, np.dot(M_np, result['true_acc'] / 9.81)) / 1000.0
        Qb_pred = np.dot(R_np.T, np.dot(M_np, result['pred_acc'] / 9.81)) / 1000.0
        # Displacement in meters with negative sign
        d_iso_true = -result['true_displ'][result['config']['N_DOF']-1, :]
        d_iso_pred = -result['pred_displ'][result['config']['N_DOF']-1, :]
       
        ax.plot(d_iso_true, Qb_true[0, :], color='#000080', linewidth=1.5)
        ax.plot(d_iso_pred, Qb_pred[0, :], '--', color="#FF0015", linewidth=1.0)
       
        ax.set_title(result['config']['seismic_input'].replace('BI-BNCS_', ''))
        ax.set_xlabel("Isolation Drift (m)")
        if col == 0:
            ax.set_ylabel("Base Shear (kN)")
       
        # Set x-axis: centered at 0, symmetrical, 5 ticks rounded to 0.05
        x_min = min(np.min(d_iso_true), np.min(d_iso_pred))
        x_max = max(np.max(d_iso_true), np.max(d_iso_pred))
        x_abs_max = round_to_nearest(max(abs(x_min), abs(x_max)), base=0.05)
        ax.set_xlim(-x_abs_max, x_abs_max)
        x_ticks = np.linspace(-x_abs_max, x_abs_max, 5)
        x_ticks = [round_to_nearest(x, base=0.05) for x in x_ticks]
        ax.set_xticks(x_ticks)
       
        # Set y-axis: centered at 0, symmetrical, 5 ticks with middle at 0, rounded to 0.05
        y_min = min(np.min(Qb_true), np.min(Qb_pred))
        y_max = max(np.max(Qb_true), np.max(Qb_pred))
        y_abs_max = round_to_nearest(max(abs(y_min), abs(y_max)), base=0.05)
        ax.set_ylim(-y_abs_max, y_abs_max)
        y_ticks = np.linspace(-y_abs_max, y_abs_max, 5)
        y_ticks = [round_to_nearest(y, base=0.05) for y in y_ticks]
        ax.set_yticks(y_ticks)
       
        # Set major and minor grid: 4x4 minor grid lines
        ax.grid(True, which='major', linestyle=':', linewidth=0.5)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.5)
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(x_abs_max / 4))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(y_abs_max / 4))
   
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()
def plot_isolation_layer_acceleration(results):
    """Plot isolation layer accelerations for CNP100, SP100, ICA140 in a single column."""
    channel_map = {6: "6"}
    ch_idx = 5 # DOF 6 (isolation layer, zero-based index)
    num_rows = 3 # One row per seismic input (CNP100, SP100, ICA140)
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)
   
    # Shared legend
    handles = [Line2D([0], [0], color="C0", lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
   
    # Map seismic inputs to row indices
    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}
   
    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue
       
        time = result['time']
        true_data = result['true_acc'][ch_idx, :] / 9.81  # Convert to g
        pred_data = result['pred_acc'][ch_idx, :] / 9.81  # Convert to g
        nrmse = calculate_nrmse(pred_data, true_data)
       
        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)
       
        # Add seismic input name inside a text box
        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_ylabel(r"$\ddot{u}_{6}$ (g)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))  # Ensure 5 uniform ticks
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])  # 2 decimal places
        ax.set_xticks(np.linspace(time[0], time[-1], 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()
def plot_isolation_layer_displacement(results):
    """Plot isolation layer displacements for CNP100, SP100, ICA140 in a single column."""
    channel_map = {6: "6"}
    ch_idx = 5 # DOF 6 (isolation layer, zero-based index)
    num_rows = 3 # One row per seismic input (CNP100, SP100, ICA140)
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)
   
    # Shared legend
    handles = [Line2D([0], [0], color='C0', lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')
   
    # Map seismic inputs to row indices
    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}
   
    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue
       
        time = result['time']
        true_data = result['true_displ'][ch_idx, :]
        pred_data = result['pred_displ'][ch_idx, :]
        nrmse = calculate_nrmse(pred_data, true_data)
       
        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)
       
        # Add seismic input name inside a text box
        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_ylabel(r"$u_{6}$ (m)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))
       
        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))  # Ensure 5 uniform ticks
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])  # 2 decimal places
        ax.set_xticks(np.linspace(time[0], time[-1], 5))
   
    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()
   
# ==============================================================================
# 5. MAIN EXECUTION
# ==============================================================================
def main():
    # Set global font style for all plots
    plt.rcParams['font.style'] = 'italic'
    plt.rcParams['font.size'] = 10
   
    all_results = []
    for config in CONFIGURATIONS:
        result = run_single_inference(config)
        if result:
            all_results.append(result)
    if not all_results:
        print("No results to plot.")
        return
   
    # --- Plotting ---
    plot_isolation_layer_acceleration(all_results)
    plot_isolation_layer_displacement(all_results)
   
    # Concatenate data for modal plots
    concatenated = {}
    separator_times = []
    concatenated["ground_motion"] = np.concatenate([r["ground_motion"] for r in all_results])
    max_modes = max(r['config']['N_MODES'] for r in all_results)
    for key in ["damp", "freq"]:
        padded_runs = []
        for r in all_results:
            run_data = r[key]
            padding = [np.full_like(run_data[0], np.nan) for _ in range(max_modes - len(run_data))]
            padded_runs.append(run_data + padding)
        concatenated[key] = [np.concatenate([run[i] for run in padded_runs]) for i in range(max_modes)]
    time_arrays = []
    current_time = 0
    for r in all_results:
        run_duration = r['time'][-1] - r['time'][0]
        time_arrays.append(np.linspace(current_time, current_time + run_duration, len(r['time'])))
        current_time += run_duration + (1/Fs)
        if len(separator_times) < len(all_results) - 1:
            separator_times.append(current_time - (1/Fs)/2)
    concatenated["time"] = np.concatenate(time_arrays)
   
    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Frequency')
    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Damping')
    plot_individual_mode_shapes(all_results)
    plot_hysteresis_comparison(all_results)
if __name__ == "__main__":
    main()

LOAD FROM FILE

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import resample
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
import os.path
# Import your custom functions (make sure these .py files are in the same directory)
from models import PhysicsInformedLSTM2, PhysicsInformedLSTM_Contributions
from utils import normalize, calculate_nrmse
from SDOF_aceleracion_promedio_tf import SDOF_aceleracion_promedio_tf
# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ==============================================================================
# 1. DEFINE ALL INFERENCE CONFIGURATIONS
# ==============================================================================
CONFIGURATIONS = [
    {
        "seismic_input": "BI-BNCS_CNP100",
        "feature_vector_filename": "lstm_features_CNP100V2.npy",
        "N_MODES": 3,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL",
        "start_sec": 2.0,
        "end_sec": 39.0,
    },
    {
        "seismic_input": "BI-BNCS_SP100",
        "feature_vector_filename": "lstm_features_SP100V2.npy",
        "N_MODES": 3,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL",
        "start_sec": 6.0,
        "end_sec": 171.0,
    },
    {
        "seismic_input": "BI-BNCS_ICA140",
        "feature_vector_filename": "lstm_features_ICA140V2.npy",
        "N_MODES": 3,
        "N_DOF": 6,
        "current_hidden_dim": 128,
        "ACC_LOSS_WEIGHT": 5,
        "DISPL_LOSS_WEIGHT": 1,
        "init_str": "FINAL",
        "start_sec": 10.0,
        "end_sec": 179.0,
    }
]

# Constants
WINDOW_SIZE = 100
Fs = 100
ORIGINAL_FS = 200
N_DOF = 6
M = tf.convert_to_tensor(np.diag([593.3, 1078.1, 1036.3, 703, 674, (1868.5 + 406.6)]).astype(np.float32))
R = tf.convert_to_tensor(np.ones((N_DOF, 1), dtype=np.float32))
M_np = M.numpy()
R_np = R.numpy()

# ==============================================================================
# 2. LOAD NPZ RESULTS FUNCTION
# ==============================================================================
def load_npz_results(config):
    """
    Loads saved results from an .npz file for a single configuration and returns results.
    """
    print("-" * 50)
    print(f"Loading results for: {config['seismic_input']}")

    # Unpack config
    seismic_input = config['seismic_input']
    N_MODES = config['N_MODES']
    N_DOF = config['N_DOF']
    current_hidden_dim = config['current_hidden_dim']

    # Generate folder and .npz file path
    weight_str = f"acc{round(100*config['ACC_LOSS_WEIGHT'])}_disp{round(100*config['DISPL_LOSS_WEIGHT'])}"
    folder_name = f"{seismic_input}_{weight_str}_NM{N_MODES}_NDOF{N_DOF}_{config['init_str']}_Hiddim{current_hidden_dim}"
    save_dir = os.path.join(os.getcwd(), folder_name)
    npz_path = os.path.join(save_dir, f"results_{folder_name}.npz")

    # Load .npz file
    try:
        data = np.load(npz_path, allow_pickle=True)
        print(f"Successfully loaded results from: {npz_path}")
    except FileNotFoundError as e:
        print(f"Error: Could not find .npz file for {seismic_input}: {e}")
        return None
    except Exception as e:
        print(f"Error loading .npz file for {seismic_input}: {e}")
        return None

    # Extract data, ensuring compatibility with plotting functions
    pred_acc_np = data['pred_acc']
    true_acc_np = data['true_acc']
    pred_displ_np = data['pred_displ']
    true_displ_np = data['true_displ']
    damp_np = data['damp']
    freq_np = data['freq']
    pred_phi_np = data['phi']
    time_axis = data['time']
    ground_motion_u = data['ground_motion']
    config = data['config'].item()  # Convert numpy object to dictionary

    return {
        "config": config,
        "pred_acc": pred_acc_np,
        "true_acc": true_acc_np,
        "pred_displ": pred_displ_np,
        "true_displ": true_displ_np,
        "damp": damp_np,
        "freq": freq_np,
        "phi": pred_phi_np,
        "time": time_axis,
        "ground_motion": ground_motion_u
    }

# ==============================================================================
# 3. REFINED PLOTTING FUNCTIONS
# ==============================================================================
def get_rounded_limits_and_ticks(data_min, data_max, num_ticks=5):
    if data_min == data_max: data_max += 1e-6
    data_range = data_max - data_min
    power = 10**np.floor(np.log10(data_range))
    possible_steps = np.array([0.05, 0.1, 0.2, 0.25, 0.5, 1.0, 2.0, 2.5, 5.0])
    best_step = possible_steps[np.argmin(np.abs(data_range / (power * possible_steps) - num_ticks))] * power
    new_min = np.floor(data_min / best_step) * best_step
    new_max = np.ceil(data_max / best_step) * best_step
    ticks = np.arange(new_min, new_max + best_step/2, best_step)
    return (new_min, new_max), ticks

def plot_concatenated_modal_params(concatenated_data, max_modes, separator_times, all_results, plot_type='Frequency'):
    fig, axs = plt.subplots(min(2, max_modes) + 1, 1, figsize=(8.5, 5.5), gridspec_kw={'height_ratios': [1, 1, 0.5]}, sharex=True)

    time = concatenated_data["time"]
    if plot_type == 'Frequency': data_key, color = "freq", '#000080'
    else: data_key, color = "damp", "#0678AD"
    plot_idx = 0
    for i in range(min(2, max_modes) - 1, -1, -1):
        ax = axs[plot_idx]
        data = concatenated_data[data_key][i]

        if plot_type == 'Damping':
            data = data * 100
            ax.set_ylabel(fr"$\xi_{{{i+1}}}$ (%)")
            ax.set_ylim(0, 50)
            ax.set_yticks(np.arange(0, 51, 10))
        else:
            ax.set_ylabel(fr"$f_{{{i+1}}}$ (Hz)")
            if i == 0:
                ax.set_ylim(0, 1.5)
                ax.set_yticks([0, 0.5, 1, 1.5])
            elif i == 1:
                ax.set_ylim(1, 5)
                ax.set_yticks([1, 2, 3, 4, 5])
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'))
        ax.plot(time, data, color=color, linewidth=1.2)
        ax.grid(True, linestyle=':')
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)
        plot_idx += 1
    ax_gm = axs[-1]
    time_offset = 0
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    for i, result in enumerate(all_results):
        run_time = result['time'] - result['time'][0] + time_offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=colors[i])
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max(concatenated_data['ground_motion'] / 9.81) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', ''), ha='center', fontsize=10)
        time_offset = run_time[-1] + (1/Fs)
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)

    ax_gm.set_xlim(0, time[-1])
    ymin, ymax = ax_gm.get_ylim()
    y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max)
    ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

def plot_individual_mode_shapes(results):
    channel_map = {1: "1", 3: "3", 6: "6"}
    channels_to_plot = [0, 2, 5]
    mode_colors = ['#6a0dad', '#20b2aa']

    fig, axs = plt.subplots(4, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1, 1, 0.5]}, sharex=True, squeeze=True)

    handles = [Line2D([0], [0], color=c, label=f'Mode {i+1}') for i, c in enumerate(mode_colors)]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')

    total_time = 0
    separator_times = []
    for i, result in enumerate(results):
        duration = len(result['time']) / Fs
        total_time += duration
        if i < len(results) - 1:
            separator_times.append(total_time - (1 / (2 * Fs)))

    for row, ch_idx in enumerate(channels_to_plot):
        ax = axs[row]
        channel_name_latex = channel_map[ch_idx + 1].replace(' ', r'\ ')
        time_offset = 0
        for i, result in enumerate(results):
            run_time = result['time'] - result['time'][0] + time_offset
            for mode_idx in range(2):
                ax.plot(run_time, result['phi'][mode_idx][ch_idx, :], color=mode_colors[mode_idx], linewidth=1.2)
            time_offset += len(result['time']) / Fs
        ax.set_ylabel(fr"$\phi_{{{channel_name_latex}}}$")
        ax.grid(True, linestyle=':')
        ax.set_xlim(0, total_time)
        ax.set_ylim(-1.5, 2.5)
        ax.set_yticks([-1.5, -0.5, 0.5, 1.5, 2.5])
        ax.set_xticks(np.linspace(0, total_time, 5))
        for sep_time in separator_times: ax.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)

    ax_gm = axs[3]
    time_offset = 0
    for i, result in enumerate(results):
        run_time = result['time'] - result['time'][0] + time_offset
        ax_gm.plot(run_time, result['ground_motion'] / 9.81, color=f"C{i}")
        text_x = run_time[0] + (run_time[-1] - run_time[0]) / 2
        text_y = np.max([r['ground_motion'].max() / 9.81 for r in results]) * 0.8
        ax_gm.text(text_x, text_y, result['config']['seismic_input'].replace('BI-BNCS_', '')[0:6], ha='center', fontsize=8)
        time_offset += len(result['time']) / Fs
    ax_gm.set_ylabel(r"$\ddot{u}_{g}$ (g)")
    ax_gm.set_xlabel("Time (s)")
    ax_gm.grid(True, linestyle=':')
    ax_gm.set_xlim(0, total_time)
    ymin, ymax = ax_gm.get_ylim()
    y_abs_max = np.ceil(max(abs(ymin), abs(ymax)))
    ax_gm.set_ylim(-y_abs_max, y_abs_max)
    ax_gm.set_yticks(np.linspace(-y_abs_max, y_abs_max, 5))
    ax_gm.set_xticks(np.linspace(0, total_time, 5))
    for sep_time in separator_times: ax_gm.axvline(x=sep_time, color='k', linestyle='--', linewidth=1.5)

    plt.tight_layout(rect=[0.15, 0, 0.85, 0.96])
    plt.show()

def round_to_nearest(value, base=0.05):
    return np.round(value / base) * base

def plot_hysteresis_comparison(results):
    global M_np, R_np
    num_events = len(results)
    fig, axs = plt.subplots(1, num_events, figsize=(3.5 * num_events, 3.5), squeeze=False, sharey=False)

    handles = [Line2D([0], [0], color='#000080', lw=2.0, label='True'),
               Line2D([0], [0], color="#FF0015", lw=1.0, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=True, edgecolor='black')

    for col, result in enumerate(results):
        ax = axs[0, col]
        Qb_true = np.dot(R_np.T, np.dot(M_np, result['true_acc'] / 9.81)) / 1000.0
        Qb_pred = np.dot(R_np.T, np.dot(M_np, result['pred_acc'] / 9.81)) / 1000.0
        d_iso_true = -result['true_displ'][result['config']['N_DOF']-1, :]
        d_iso_pred = -result['pred_displ'][result['config']['N_DOF']-1, :]

        ax.plot(d_iso_true, Qb_true[0, :], color='#000080', linewidth=1.5)
        ax.plot(d_iso_pred, Qb_pred[0, :], '--', color="#FF0015", linewidth=1.0)

        ax.set_title(result['config']['seismic_input'].replace('BI-BNCS_', ''))
        ax.set_xlabel("Isolation Drift (m)")
        if col == 0:
            ax.set_ylabel("Base Shear (kN)")

        x_min = min(np.min(d_iso_true), np.min(d_iso_pred))
        x_max = max(np.max(d_iso_true), np.max(d_iso_pred))
        x_abs_max = round_to_nearest(max(abs(x_min), abs(x_max)), base=0.05)
        ax.set_xlim(-x_abs_max, x_abs_max)
        x_ticks = np.linspace(-x_abs_max, x_abs_max, 5)
        x_ticks = [round_to_nearest(x, base=0.05) for x in x_ticks]
        ax.set_xticks(x_ticks)

        y_min = min(np.min(Qb_true), np.min(Qb_pred))
        y_max = max(np.max(Qb_true), np.max(Qb_pred))
        y_abs_max = round_to_nearest(max(abs(y_min), abs(y_max)), base=0.05)
        ax.set_ylim(-y_abs_max, y_abs_max)
        y_ticks = np.linspace(-y_abs_max, y_abs_max, 5)
        y_ticks = [round_to_nearest(y, base=0.05) for y in y_ticks]
        ax.set_yticks(y_ticks)

        ax.grid(True, which='major', linestyle=':', linewidth=0.5)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.5)
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(x_abs_max / 4))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(y_abs_max / 4))

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()

def plot_isolation_layer_acceleration(results):
    channel_map = {6: "6"}
    ch_idx = 5
    num_rows = 3
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)

    handles = [Line2D([0], [0], color="C0", lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')

    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}

    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue

        time = result['time']
        true_data = result['true_acc'][ch_idx, :] / 9.81
        pred_data = result['pred_acc'][ch_idx, :] / 9.81
        nrmse = calculate_nrmse(pred_data, true_data)

        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)

        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))

        ax.set_ylabel(r"$\ddot{u}_{6}$ (g)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))

        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])
        ax.set_xticks(np.linspace(time[0], time[-1], 5))

    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()

def plot_isolation_layer_displacement(results):
    channel_map = {6: "6"}
    ch_idx = 5
    num_rows = 3
    fig, axs = plt.subplots(num_rows, 1, figsize=(8.5, 6), sharex=False, squeeze=True)

    handles = [Line2D([0], [0], color='C0', lw=2.0, label='True'),
               Line2D([0], [0], color="C1", lw=1.2, linestyle='--', label='Predicted')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2, frameon=True, edgecolor='black')

    seismic_order = ['CNP100', 'SP100', 'ICA140']
    result_map = {result['config']['seismic_input'].replace('BI-BNCS_', ''): result for result in results}

    for row, seismic_input in enumerate(seismic_order):
        ax = axs[row]
        result = result_map.get(seismic_input)
        if result is None:
            print(f"Warning: No data for {seismic_input}")
            continue

        time = result['time']
        true_data = result['true_displ'][ch_idx, :]
        pred_data = result['pred_displ'][ch_idx, :]
        nrmse = calculate_nrmse(pred_data, true_data)

        ax.plot(time, true_data, color="C0", linewidth=2.0)
        ax.plot(time, pred_data, '--', color="C1", linewidth=1.2)

        ax.text(0.05, 0.95, seismic_input, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))

        ax.set_ylabel(r"$u_{6}$ (m)")
        ax.set_xlabel("Time (s)")
        ax.grid(True, linestyle=':')
        ax.text(0.95, 0.95, f"NRMSE: {nrmse:.2f}", transform=ax.transAxes, ha='right', va='top',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=0.2))

        ax.set_xlim(time[0], time[-1])
        ylim, yticks = get_rounded_limits_and_ticks(min(np.min(true_data), np.min(pred_data)),
                                                    max(np.max(true_data), np.max(pred_data)))
        ax.set_ylim(ylim)
        ax.set_yticks(np.linspace(ylim[0], ylim[1], 5))
        ax.set_yticklabels([f'{y:.2f}' for y in np.linspace(ylim[0], ylim[1], 5)])
        ax.set_xticks(np.linspace(time[0], time[-1], 5))

    plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=0.5)
    plt.show()

# ==============================================================================
# 5. MAIN EXECUTION
# ==============================================================================
def main():
    plt.rcParams['font.style'] = 'italic'
    plt.rcParams['font.size'] = 10

    all_results = []
    for config in CONFIGURATIONS:
        result = load_npz_results(config)
        if result:
            all_results.append(result)
    if not all_results:
        print("No results to plot.")
        return

    plot_isolation_layer_acceleration(all_results)
    plot_isolation_layer_displacement(all_results)

    # Concatenate data for modal plots
    concatenated = {}
    separator_times = []
    concatenated["ground_motion"] = np.concatenate([r["ground_motion"] for r in all_results])
    max_modes = max(r['config']['N_MODES'] for r in all_results)
    for key in ["damp", "freq"]:
        padded_runs = []
        for r in all_results:
            run_data = r[key]
            if len(run_data) < max_modes:
                padding = [np.full_like(run_data[0], np.nan) for _ in range(max_modes - len(run_data))]
                padded_runs.append(run_data + padding)
            else:
                padded_runs.append(run_data)  # No padding needed
        concatenated[key] = [np.concatenate([run[i] for run in padded_runs]) for i in range(max_modes)]
    time_arrays = []
    current_time = 0
    for r in all_results:
        run_duration = r['time'][-1] - r['time'][0]
        time_arrays.append(np.linspace(current_time, current_time + run_duration, len(r['time'])))
        current_time += run_duration + (1/Fs)
        if len(separator_times) < len(all_results) - 1:
            separator_times.append(current_time - (1/Fs)/2)
    concatenated["time"] = np.concatenate(time_arrays)

    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Frequency')
    plot_concatenated_modal_params(concatenated, max_modes, separator_times, all_results, plot_type='Damping')
    plot_individual_mode_shapes(all_results)
    plot_hysteresis_comparison(all_results)

if __name__ == "__main__":
    main()